In [0]:

emp_data = [
    ["001","101","John Doe","30","Male","50000","2015-01-01"],
    ["002","101","Jane Smith","25","Female","45000","2016-02-15"],
    ["003","102","Bob Brown","35","Male","55000","2014-05-01"],
    ["004","102","Alice Lee","28","Female","48000","2017-09-30"],
    ["005","103","Jack Chan","40","Male","60000","2013-04-01"],
    ["006","103","Jill Wong","32","Female","52000","2018-07-01"],
    ["007","101","James Johnson","42","Male","70000","2012-03-15"],
    ["008","102","Kate Kim","29","Female","51000","2019-10-01"],
    ["009","103","Tom Tan","33","Male","58000","2016-06-01"],
    ["010","104","Lisa Lee","27","Female","47000","2018-08-01"],
    ["011","104","David Park","38","Male","65000","2015-11-01"],
    ["012","105","Susan Chen","31","Female","54000","2017-02-15"],
    ["013","106","Brian Kim","45","Male","75000","2011-07-01"],
    ["014","107","Emily Lee","26","Female","46000","2019-01-01"],
    ["015","106","Michael Lee","37","Male","63000","2014-09-30"],
    ["016","107","Kelly Zhang","30","Female","49000","2018-04-01"],
    ["017","105","George Wang","34","Male","57000","2016-03-15"],
    ["018","104","Nancy Liu","29","","50000","2017-06-01"],
    ["019","103","Steven Chen","36","Male","62000","2015-08-01"],
    ["020","102","Grace Kim","32","Female","53000","2018-11-01"]
]

emp_schema = "employee_id string, department_id string, name string, age string, gender string, salary string, hire_date string"

dept_data = [
    ["101", "Sales", "NYC", "US", "1000000"],
    ["102", "Marketing", "LA", "US", "900000"],
    ["103", "Finance", "London", "UK", "1200000"],
    ["104", "Engineering", "Beijing", "China", "1500000"],
    ["105", "Human Resources", "Tokyo", "Japan", "800000"],
    ["106", "Research and Development", "Perth", "Australia", "1100000"],
    ["107", "Customer Service", "Sydney", "Australia", "950000"]
]

dept_schema = "department_id string, department_name string, city string, country string, budget string"

In [0]:

emp = spark.createDataFrame(data=emp_data, schema=emp_schema)
dept = spark.createDataFrame(data=dept_data, schema=dept_schema)

In [0]:
emp.show()
dept.show()

In [0]:
# Get number of partitions for emp
emp.rdd.getNumPartitions()

In [0]:
df_joined=emp.alias("e").join(dept.alias("d"),how="inner",on=emp.department_id==dept.department_id)
df_joined.show()

In [0]:
df_joined.select(
    "e.name",
    "e.department_id",
    "d.department_name",
    "e.salary"
).show()

In [0]:
df_joined = emp.alias("e").join(dept.alias("d"), how="left_outer", on=emp.department_id==dept.department_id)

In [0]:
df_joined.select("e.name", "d.department_name", "d.department_id", "e.salary").show()

In [0]:
df_final = emp.join(dept, how="left_outer", 
                   on=(emp.department_id==dept.department_id) & ((emp.department_id == "101") | (emp.department_id == "102")) 
                    & (emp.salary.isNull())
                   )

In [0]:
df_final.show()


In [0]:
# Create a list
data = ['buying books at amazom.com', 'rameses@egypt.com', 'matt@t.co', 'narendra@modi.com']
# Convert the list to DataFrame
df = spark.createDataFrame(data, "string")
df.show(truncate =False)

 How to filter valid emails from a list?

In [0]:
df.filter(df["value"])

In [0]:
# Define a regular expression pattern for emails
from pyspark.sql import functions as F
pattern = "^[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+$"
# Apply filter operation to keep only valid emails
df_filtered = df.filter(F.col("value").rlike(pattern))
# Show the DataFrame
df_filtered.show()

In [0]:
# Sample data
data = [
(2021, 1, "US", 5000),
(2021, 1, "EU", 4000),
(2021, 2, "US", 5500),
(2021, 2, "EU", 4500),
(2021, 3, "US", 6000),
(2021, 3, "EU", 5000),
(2021, 4, "US", 7000),
(2021, 4, "EU", 6000),
]
# Create DataFrame
columns = ["year", "quarter", "region", "revenue"]
df = spark.createDataFrame(data, columns)
df.show()

Convert region categories to Columns and sum the revenue

In [0]:
from pyspark.sql.functions import *
df_g=df.groupBy("region").agg(sum(df["revenue"]))
df_g.show()

In [0]:
df_final=df_g.withColumn("US",when(df_g["region"]=='US',df_g["sum(revenue)"])).withColumn('EU',when(df_g["region"]=='EU',df_g["sum(revenue)"])).drop("region","sum(revenue)")
df_final.show()

In [0]:

from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.functions import *
from pyspark.sql.types import *

df.groupBy().pivot("region").count().show()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

pivoted_df = df.groupBy("year", "quarter").pivot("region").agg(sum("revenue"))
display(pivoted_df)

How to get the mean of a variable grouped by another variable?

In [0]:
# Sample data
data = [("1001", "Laptop", 1000),
("1002", "Mouse", 50),
("1003", "Laptop", 1200),
("1004", "Mouse", 30),
("1005", "Smartphone", 700)]
# Create DataFrame
columns = ["OrderID", "Product", "Price"]
df = spark.createDataFrame(data, columns)
df.show()

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import avg

w = Window.partitionBy(df["Product"])
final_df = df.withColumn("avg_price", avg(df["Price"]).over(w))
display(final_df)

In [0]:
from pyspark.sql.functions import mean
# GroupBy and aggregate
result = df.groupBy("Product").agg(mean("Price").alias("Total_Sales"))
# Show results
result.show()

. How to compute the euclidean distance between two columns?

In [0]:
# Define your series
data = [(1, 10), (2, 9), (3, 8), (4, 7), (5, 6), (6, 5), (7, 4), (8, 3), (9, 2), (10, 1)]
# Convert list to DataFrame
df = spark.createDataFrame(data, ["series1", "series2"])
df.show()

In [0]:
# Calculate squared differences
df = df.withColumn("squared_diff", expr("POW(series1 - series2, 2)"))
# Sum squared differences and take square root
df.agg(expr("SQRT(SUM(squared_diff))").alias("euclidean_distance")).show()
